In [1]:
import faiss
import json
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Load the chunked data (Metadata)
with open('ww2_chunks.json', 'r') as f:
    chunks_data = json.load(f)

# 2. Load the FAISS Index
index = faiss.read_index("ww2_index.faiss")

# 3. Initialize the same model used for indexing
model = SentenceTransformer('all-MiniLM-L6-v2')

def retrieve_top_k(query, k=5):
    # 4. Encode the query into a vector
    # We wrap query in a list because the model expects a batch
    query_vector = model.encode([query])
    
    # 5. Normalize the query vector
    # This is required because we used IndexFlatIP (Inner Product) 
    # for Cosine Similarity during the indexing phase
    faiss.normalize_L2(query_vector)
    
    # 6. Search the index
    # distances (D): similarity scores (1.0 is a perfect match)
    # indices (I): the position of the match in our original chunks_data list
    distances, indices = index.search(query_vector, k)
    
    # 7. Map results back to the original JSON metadata
    results = []
    for i, idx in enumerate(indices[0]):
        # index.search returns -1 if no match is found for a slot
        if idx != -1:
            chunk = chunks_data[idx]
            results.append({
                "score": float(distances[0][i]),
                "title": chunk['title'],
                "url": chunk['url'],
                "content": chunk['content']
            })
    return results

# --- EXECUTION ---
hardcoded_query = "What were the major turning points in the Pacific Theater?"
top_results = retrieve_top_k(hardcoded_query, k=3)

print(f"Query: {hardcoded_query}\n")
for i, res in enumerate(top_results):
    print(f"Result {i+1} (Score: {res['score']:.4f})")
    print(f"Source: {res['title']} | {res['url']}")
    print(f"Text: {res['content'][:200]}...")
    print("-" * 30)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: What were the major turning points in the Pacific Theater?

Result 1 (Score: 0.5150)
Source: List of naval and land-based operations in the Pacific Theater during World War II | https://en.wikipedia.org/wiki/List_of_naval_and_land-based_operations_in_the_Pacific_Theater_during_World_War_II
Text: List of codenames of naval and land based operations in the Pacific Theater during World War II including Japan, Oceania, and the Pacific Rim....
------------------------------
Result 2 (Score: 0.4851)
Source: Events preceding World War II in Asia | https://en.wikipedia.org/wiki/Events_preceding_World_War_II_in_Asia
Text: This article is concerned with the events that preceded World War II in Asia.

Noteworthy events
The following events played a significant role in setting the stage for the involvement of Asia and the...
------------------------------
Result 3 (Score: 0.4556)
Source: List of theaters and campaigns of World War II | https://en.wikipedia.org/wiki/List_of_theaters_and_camp